In [1]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, YEARS, RESULTS_DIR, SRC_DIR, SRC_DIRS, CHOC_SCENARIO

CONTENT = "dir"        # "dir" : rd_dir_CI | "tot" : rd_tot_CI
YEAR = 2022 # YEARS[0]       # année de la table de base (la même pour tous les horizons)
YEARS_CALC = list(range(2025, 2051))                   # années calculées : le choc de l'année t est l'écart en niveau vs la baseline de l'année t (rien à cumuler)
DISPLAY_YEARS = [2025, 2030, 2035, 2040, 2045, 2050]   # années affichées dans le tableau (parmi YEARS_CALC) ; YEAR = année de base, variation 0
OMEGA_FILE = "omega_progressif.csv"                               # inputs/ : origin, dest, sector, delta_pct, + colonne year pour un ω par année (sans colonne year : le même ω chaque année)
UNIT = "pct"                                           # tableau : "pct" = variation du coût unitaire en % (exp(dlnCost) - 1) | "log" = dlnCost

EU27 = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT",
        "RO", "SK", "SI", "ES", "SE"]
BLOCKS = {
    "USA": ["US"],
    "EU27": EU27,
    "RestOECD": ["AU", "CA", "JP", "KR", "MX", "NZ", "NO", "CH", "UK"],
    "China": ["CN"],
    "RestBRICS": ["BR", "RU", "IN", "ZA"],
    "RestAsia": ["BD", "KH", "ID", "LA", "MY", "MM", "NP", "PK", "PH", "SG", "TW", "TJ", "TL", "TH", "VN"],
}
REGION_TO_BLOCK = {region: block for block, regions in BLOCKS.items() for region in regions}
DEFAULT_BLOCK = "RoW"

MANUFACTURING = ["C13_C15", "C16_C18", "C20+C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31_C33"]
NONMANUF = "NONMANUF"

In [2]:
def read_matrix(path):
    return pd.read_csv(path, sep=";", header=[0, 1, 2], index_col=[0, 1, 2])


def flat(m):
    m = m.copy()
    m.index, m.columns = m.index.droplevel(1), m.columns.droplevel(1)
    m.index.names = m.columns.names = ["Region", "Sector"]
    return m


MODEL_BLOCKS = {
    2: {"D": (None, True), "F": (None, False)},
    3: {"I": (True, None), "X": (False, None)},
    4: {"same_same": (True, True), "same_diff": (True, False), "diff_same": (False, True), "diff_diff": (False, False)},
}


def block_matrices(m, region_to_block=REGION_TO_BLOCK, default_block=DEFAULT_BLOCK):
    reg_i, reg_j = m.index.get_level_values("Region"), m.columns.get_level_values("Region")
    foreign = np.asarray(reg_i)[:, None] != np.asarray(reg_j)[None, :]
    rows = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_i], m.index.get_level_values("Sector")], names=["Bloc", "Sector"])
    cols = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_j], m.columns.get_level_values("Sector")], names=["Bloc", "Sector"])
    out = []
    for mask in (~foreign, foreign):
        a = pd.DataFrame(m.to_numpy() * mask, index=rows, columns=cols)
        out.append(a.groupby(level=["Bloc", "Sector"]).sum().T.groupby(level=["Bloc", "Sector"]).sum().T)
    return out


def omega_blocks(index, columns, omega, manufacturing=MANUFACTURING):
    b_i, sec_i = np.asarray(index.get_level_values("Bloc")), np.asarray(index.get_level_values("Sector"))
    b_j = np.asarray(columns.get_level_values("Bloc"))
    om = np.ones((len(index), len(columns)))
    for r in omega.sort_values("sector", key=lambda s: s != "ALL").itertuples():
        rows = (b_i == r.origin) & ((sec_i == r.sector) | (r.sector == "ALL"))
        om[np.ix_(rows, b_j == r.dest)] = 1 + r.delta_pct / 100
    return np.where(np.isin(sec_i, manufacturing)[:, None], om, 1.0)


def shock_blocks(cd, cf, zd, zf, omega, beta, model, manufacturing=MANUFACTURING):
    idx, cols = cd.index, cd.columns
    bloc_j, sec_j = cols.get_level_values("Bloc"), np.asarray(cols.get_level_values("Sector"))
    same_sec = np.asarray(idx.get_level_values("Sector"))[:, None] == sec_j[None, :]
    om = omega_blocks(idx, cols, omega, manufacturing)
    cdv, cfv = cd.to_numpy(), cf.to_numpy()
    s0, s1 = {}, {}
    for k, (sec, cty) in MODEL_BLOCKS[model].items():
        ms = np.ones_like(same_sec) if sec is None else same_sec == sec
        d = (cdv * ms).sum(0) if cty is not False else 0.0                     # partie domestique (jamais choquée)
        s0[k] = d + ((cfv * ms).sum(0) if cty is not True else 0.0)
        s1[k] = d + ((cfv * om * ms).sum(0) if cty is not True else 0.0)

    none = np.zeros(len(sec_j))
    cases = [("A", none, none)]
    if zd is not None:                 # scénario B : a besoin des volumes z
        by_sector = lambda a: pd.DataFrame(a, index=idx, columns=cols).groupby(level="Sector").sum()
        c_dom, z_dom = by_sector(cdv), by_sector(zd.to_numpy())
        lost = by_sector((1 - om) * zf.to_numpy())
        rho_d = (c_dom / z_dom.where(z_dom > 0)).fillna(0.0)
        gain = np.maximum(rho_d * lost, -c_dom)
        g_same = gain.to_numpy()[gain.index.get_indexer(sec_j), np.arange(len(sec_j))]
        cases.append(("B", g_same, gain.sum().to_numpy() - g_same))

    out = []
    for scenario, g_same, g_diff in cases:
        res = pd.DataFrame({"bloc": bloc_j, "sector": sec_j})
        for k, (sec, cty) in MODEL_BLOCKS[model].items():
            g = 0.0 if cty is False else (g_same + g_diff if sec is None else g_same if sec else g_diff)
            with np.errstate(all="ignore"):
                res[f"S_{k}"], res[f"dlnS_{k}"] = s0[k], np.where(s0[k] > 0, np.log((s1[k] + g) / s0[k]), 0.0)
        out.append(res.assign(scenario=scenario))
    res = pd.concat(out, ignore_index=True)
    if beta is not None:
        res = res.merge(beta, on="sector", how="left")
        res["dlnCost"] = sum(res[f"beta_{k}"] * res[f"dlnS_{k}"] for k in MODEL_BLOCKS[model])
    return res


def add_sector(sect, res, components, beta=None):
    sub = res[res["sector"].str.startswith(sect)]
    keys = [sub["bloc"], sub["scenario"]]
    agg = pd.DataFrame(index=sub.groupby(keys, sort=False).size().index)
    for k in components:
        s0 = sub[f"S_{k}"].groupby(keys, sort=False).sum()
        s1 = (sub[f"S_{k}"] * np.exp(sub[f"dlnS_{k}"])).groupby(keys, sort=False).sum()      # stock après choc
        with np.errstate(all="ignore"):
            agg[f"S_{k}"], agg[f"dlnS_{k}"] = s0, np.where(s0 > 0, np.log(s1 / s0), 0.0)
    agg = agg.reset_index().assign(sector=sect)
    if beta is not None:
        agg = agg.merge(beta, on="sector", how="left")
        agg["dlnCost"] = sum(agg[f"beta_{k}"] * agg[f"dlnS_{k}"] for k in components)
    return pd.concat([res, agg[res.columns]], ignore_index=True)


def sum_sectors(res, components, beta=None):
    keys = [res["sector"], res["scenario"]]
    out = pd.DataFrame(index=res.groupby(keys, sort=False).size().index)
    for k in components:
        s0 = res[f"S_{k}"].groupby(keys, sort=False).sum()
        s1 = (res[f"S_{k}"] * np.exp(res[f"dlnS_{k}"])).groupby(keys, sort=False).sum()      # stock après choc
        with np.errstate(all="ignore"):
            out[f"S_{k}"], out[f"dlnS_{k}"] = s0, np.where(s0 > 0, np.log(s1 / s0), 0.0)
    out = out.reset_index()
    out = out[["sector"] + [c_ for k in components for c_ in (f"S_{k}", f"dlnS_{k}")] + ["scenario"]]
    if beta is not None:
        out = out.merge(beta, on="sector", how="left")
        out["dlnCost"] = sum(out[f"beta_{k}"] * out[f"dlnS_{k}"] for k in components)
    return out


In [3]:
if BASE not in ("EXIOBASE", "FIGARO", "OECD_ICIO", "GTAP"):
    raise ValueError(f"BASE={BASE!r} : le choc utilise les flux Z d'une base MRIO (EXIOBASE, FIGARO, OECD_ICIO ou GTAP)")


Z_REF_YEAR = 2010   # année dont z_CI sert à reconstruire les volumes des années sans z_CI (coefficients techniques fixes)


def mpath(name, year, suffix="_agg"):
    return Path(SRC_DIR) / BASE / f"output_loop_{BASE}_{year}" / f"{name}_{BASE}_{year}{suffix}.csv"


prod = pd.read_csv(Path(SRC_DIR) / BASE / f"output_loop_{BASE}" / f"production_{BASE}_agg.csv", sep=";").set_index(["Region", "Sector"])
_cache = {}


def load_z(c, year):
    if mpath("z_CI", year).exists():
        return flat(read_matrix(mpath("z_CI", year))).reindex(index=c.index, columns=c.columns).fillna(0.0), "z_CI exact"
    if "A" not in _cache:
        for d in (SRC_DIR, PATH_WRK + "SRDV (c,s)/", PATH_WRK + "SRDQ (c,s)/"):   # z_CI de l'année de référence, dans n'importe quel dossier de run
            p = Path(d) / BASE / f"output_loop_{BASE}_{Z_REF_YEAR}" / f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv"
            if p.exists():
                z_ref = flat(read_matrix(p)).reindex(index=c.index, columns=c.columns).fillna(0.0)
                x_ref = prod[str(Z_REF_YEAR)].reindex(c.index).fillna(0.0).to_numpy()
                _cache["A"] = z_ref.to_numpy() / np.where(x_ref > 0, x_ref, 1.0)[None, :]
                break
        else:
            raise FileNotFoundError(f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv introuvable : lancer know_calc_loop4.0 pour l'année {Z_REF_YEAR}")
    x = prod[str(year)].reindex(c.index).fillna(0.0).to_numpy()
    z_prox = _cache["A"] * x[None, :]                                    # volumes avec les coefficients techniques de Z_REF_YEAR
    zs, cs = z_prox.sum(axis=1), c.to_numpy().sum(axis=1)
    rho = np.divide(cs, zs, out=np.zeros(len(zs)), where=zs > 0)         # R&D du fournisseur par euro vendu cette année
    z = np.where(rho[:, None] > 0, np.divide(c.to_numpy(), rho[:, None], out=np.zeros(c.shape), where=rho[:, None] > 0), z_prox)   # z = c / rho
    return pd.DataFrame(z, index=c.index, columns=c.columns), f"z reconstruit (c/rho, A {Z_REF_YEAR})"


def find(name):
    for d in SRC_DIRS:
        p = Path(d) / BASE / f"output_loop_{BASE}_{YEAR}" / f"{name}_{BASE}_{YEAR}_agg.csv"
        if p.exists():
            return p
    raise FileNotFoundError(f"{name}_{BASE}_{YEAR}_agg.csv introuvable dans {SRC_DIRS} : lancer know_calc_loop puis aggregate_to_common_mapping")


inputs = Path(PATH_WRK) / "choc_tfp" / "inputs"
omega_all = pd.read_csv(inputs / OMEGA_FILE)                                      # ω de toutes les années (colonne year facultative)

prefix = {"dir": "rd_dir_CI", "tot": "rd_tot_CI"}[CONTENT]
c0 = flat(read_matrix(find(prefix)))
if CHOC_SCENARIO == "B":                      # seul le scénario B a besoin des volumes z (le A n'en a pas besoin)
    z0, z_src = load_z(c0, YEAR)
else:
    z0, z_src = None, "scénario A : pas de volumes z"
print(f"volumes {YEAR} : {z_src}")

cd, cf = block_matrices(c0)                       # contenu domestique / étranger, agrégé en (bloc, secteur) AVANT le choc
zd, zf = (None, None) if z0 is None else [m.reindex(index=cd.index, columns=cd.columns).fillna(0.0) for m in block_matrices(z0)]
print(f"{cd.shape[0]} (bloc, secteur) fournisseurs x {cd.shape[1]} (bloc, secteur) acheteurs")

out = Path(RESULTS_DIR) / "choc_tfp"
out.mkdir(parents=True, exist_ok=True)


volumes 2022 : z_CI exact
238 (bloc, secteur) fournisseurs x 238 (bloc, secteur) acheteurs


In [4]:
def omega_of(year):
    if "year" not in omega_all.columns:
        return omega_all
    om = omega_all[omega_all["year"] == year].drop(columns="year")
    if om.empty:
        raise ValueError(f"{OMEGA_FILE} : aucune ligne pour l'année {year}")
    return om


results = {}
for model in MODEL_BLOCKS:
    f_beta = inputs / ("beta.csv" if model == 2 else f"beta_modele{model}.csv")
    beta_m = pd.read_csv(f_beta) if f_beta.exists() else None
    cache, frames, frames_s = {}, [], []
    for year in YEARS_CALC:
        omega = omega_of(year)
        key = omega.to_csv(index=False)
        if key not in cache:
            res_m = add_sector("C", shock_blocks(cd, cf, zd, zf, omega, beta_m, model), list(MODEL_BLOCKS[model]), beta_m)
            res_m = add_sector("H", shock_blocks(cd, cf, zd, zf, omega, beta_m, model), list(MODEL_BLOCKS[model]), beta_m)
            cache[key] = (res_m, sum_sectors(res_m, list(MODEL_BLOCKS[model]), beta_m))
        frames.append(cache[key][0].assign(year=year))
        frames_s.append(cache[key][1].assign(year=year))
    results[model] = pd.concat(frames, ignore_index=True)
    results[model].to_csv(out / f"resultats{model}_blocs_{BASE}_{CONTENT}_horizon_choc{CHOC_SCENARIO}.csv", index=False)
    pd.concat(frames_s, ignore_index=True).to_csv(out / f"resultats{model}_blocs_secteurs_{BASE}_{CONTENT}_horizon_choc{CHOC_SCENARIO}.csv", index=False)
    print(f"modèle ({model}) : {len(YEARS_CALC)} années, {len(results[model])} lignes (blocs) écrites dans {out}")

modèle (2) : 26 années, 12740 lignes (blocs) écrites dans D:\ENTICE\rendu_fin_stage\runs\SRDQ (c,s)\choc_tfp
modèle (3) : 26 années, 12740 lignes (blocs) écrites dans D:\ENTICE\rendu_fin_stage\runs\SRDQ (c,s)\choc_tfp
modèle (4) : 26 années, 12740 lignes (blocs) écrites dans D:\ENTICE\rendu_fin_stage\runs\SRDQ (c,s)\choc_tfp


In [5]:
# Tableau des années affichées : une ligne par (scénario, secteur, bloc), une colonne par année. Variation du coût unitaire par rapport à la baseline
# (l'année de base ne bouge pas : 0)
tables = {}
for model in MODEL_BLOCKS:
    wide = results[model].pivot_table(index=["scenario", "sector", "bloc"], columns="year", values="dlnCost").reindex(columns=DISPLAY_YEARS)
    wide.insert(0, YEAR, 0.0)
    if UNIT == "pct":
        wide = (np.exp(wide) - 1) * 100
    wide.columns.name = None
    tables[model] = wide.round(4).reset_index()
    path = out / f"variation_cout_{BASE}_{CONTENT}_modele{model}_base{YEAR}.csv"
    tables[model].to_csv(path, index=False)
    print(f"modèle ({model}) : tableau de {len(tables[model])} lignes écrit dans {path}")

tables[2]

modèle (2) : tableau de 490 lignes écrit dans D:\ENTICE\rendu_fin_stage\runs\SRDQ (c,s)\choc_tfp\variation_cout_GTAP_dir_modele2_base2022.csv
modèle (3) : tableau de 490 lignes écrit dans D:\ENTICE\rendu_fin_stage\runs\SRDQ (c,s)\choc_tfp\variation_cout_GTAP_dir_modele3_base2022.csv
modèle (4) : tableau de 490 lignes écrit dans D:\ENTICE\rendu_fin_stage\runs\SRDQ (c,s)\choc_tfp\variation_cout_GTAP_dir_modele4_base2022.csv


,scenario,sector,bloc,2022,2025,2030,2035,2040,2045,2050
0,A,A,China,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,A,A,EU27,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,A,A,RestAsia,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,A,A,RestBRICS,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,A,A,RestOECD,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
...,...,...,...,...,...,...,...,...,...,...
485,B,R+S+T,RestAsia,0.0,0.0213,0.1292,0.2395,0.3522,0.4675,0.5855
486,B,R+S+T,RestBRICS,0.0,0.0138,0.0837,0.1548,0.2272,0.3009,0.3759
487,B,R+S+T,RestOECD,0.0,0.0154,0.0934,0.1727,0.2534,0.3354,0.4189
488,B,R+S+T,RoW,0.0,0.0204,0.1237,0.2286,0.3352,0.4435,0.5537
